In [1]:
import torch
import numpy as np
# import pandas as pd

In [3]:
torch.__version__

'2.11.0+cpu'

In [6]:
t1 = torch.tensor([
	[1,4,7],
	[2,3,6]
])

t2 = torch.tensor([
	[1.0,4,7],
	[2,3,6]
], dtype=torch.float64)
t1+t2

tensor([[ 2.,  8., 14.],
        [ 4.,  6., 12.]], dtype=torch.float64)

In [9]:
t1 = torch.FloatTensor([
	[1,4,7],
	[2,3,6]
])
t2 = torch.FloatTensor([
	[1.0,4,7],
	[2,3,6]
])
t1+t2

tensor([[ 2.,  8., 14.],
        [ 4.,  6., 12.]])

In [10]:
arr1 = np.array([
	[1,4,7],
	[2,3,6]
])

torch.from_numpy(arr1)

tensor([[1, 4, 7],
        [2, 3, 6]])

In [11]:
arr1[0,1]

np.int64(4)

In [13]:
t1[0,1]

tensor(4.)

In [14]:
def sigmoid(z):
	return 1 / (1+np.exp(-z))
sigmoid(arr1)

array([[0.73105858, 0.98201379, 0.99908895],
       [0.88079708, 0.95257413, 0.99752738]])

In [15]:
t1

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [16]:
t1.sigmoid()

tensor([[0.7311, 0.9820, 0.9991],
        [0.8808, 0.9526, 0.9975]])

In [17]:
if torch.cuda.is_available():
	device = "cuda"
else: 
	device = "cpu"

device

'cpu'

In [18]:
t1.to(device)
t2.to(device)
t1+t2

tensor([[ 2.,  8., 14.],
        [ 4.,  6., 12.]])

In [19]:
x = torch.tensor(5.0, requires_grad=True)
x

tensor(5., requires_grad=True)

In [21]:
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [22]:
f.backward()

In [23]:
x.grad

tensor(10.)

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = housing['data']
y = housing['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, train_size=0.5)

In [28]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

In [33]:
means = X_train.mean(axis=0, keepdim=True)
std = X_train.std(axis=0, keepdim=True)

In [34]:
# from sklearn.preprocessing import StandardScaler
# StandardScaler()

In [35]:
X_train = (X_train - means) / std
X_test = (X_test - means) / std
X_valid = (X_valid - means) / std

In [36]:
X_train

tensor([[ 0.8734, -1.7891, -0.2651,  ..., -0.0669, -0.5549, -0.0888],
        [-1.0794,  1.8529, -0.8767,  ..., -0.0052, -0.7516,  0.6643],
        [ 1.1752, -0.7598,  0.9239,  ..., -0.0097, -0.7047,  0.9536],
        ...,
        [ 0.8135, -0.1264, -0.1916,  ..., -0.0204,  1.6461, -1.0266],
        [-0.0534, -1.7891,  0.2515,  ..., -0.0075, -0.4987,  0.7142],
        [ 1.1353, -1.4724,  0.3872,  ..., -0.0378,  1.5056, -0.7522]])

In [45]:
# y_train[:, np.newaxis]
y_train = torch.FloatTensor(y_train).view(-1,1)
y_test = torch.FloatTensor(y_test).view(-1,1)
y_valid = torch.FloatTensor(y_valid).view(-1,1)

In [46]:
y_train

tensor([[2.6380],
        [3.1250],
        [2.5410],
        ...,
        [0.7180],
        [1.6760],
        [2.2010]])

In [48]:
n_rows, n_features = X_train.shape
n_rows, n_features

(16512, 8)

In [51]:
from torch import nn

In [52]:
help(nn.Linear)

Help on class Linear in module torch.nn.modules.linear:

class Linear(torch.nn.modules.module.Module)
 |  Linear(in_features: int, out_features: int, bias: bool = True, device=None, dtype=None) -> None
 |
 |  Applies an affine linear transformation to the incoming data: :math:`y = xA^T + b`.
 |
 |  This module supports :ref:`TensorFloat32<tf32_on_ampere>`.
 |
 |  On certain ROCm devices, when using float16 inputs this module will use :ref:`different precision<fp16_on_mi200>` for backward.
 |
 |  Args:
 |      in_features: size of each input sample
 |      out_features: size of each output sample
 |      bias: If set to ``False``, the layer will not learn an additive bias.
 |          Default: ``True``
 |
 |  Shape:
 |      - Input: :math:`(*, H_\text{in})` where :math:`*` means any number of
 |        dimensions including none and :math:`H_\text{in} = \text{in\_features}`.
 |      - Output: :math:`(*, H_\text{out})` where all but the last dimension
 |        are the same shape as the i

In [147]:
torch.manual_seed(123)

model = nn.Sequential(
	nn.Linear(8, 50), 
	nn.Sigmoid(),
	nn.Linear(50, 40),
	nn.Sigmoid(),
	nn.Linear(40, 1)
)

# sum([p.numel() for p in model.parameters()])

learning_rate = 0.1
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [148]:
n_epochs = 100

for epoch in range(n_epochs):
	y_pred = model(X_train)
	loss = loss_function(y_pred, y_train)
	loss.backward()
	optimizer.step()
	optimizer.zero_grad()
	print(f"epoch: {epoch+1}/{n_epochs}, loss: {round(loss.item(), 3)}")

epoch: 1/100, loss: 3.849
epoch: 2/100, loss: 5.739
epoch: 3/100, loss: 8.723
epoch: 4/100, loss: 6.167
epoch: 5/100, loss: 4.137
epoch: 6/100, loss: 1.542
epoch: 7/100, loss: 1.352
epoch: 8/100, loss: 1.321
epoch: 9/100, loss: 1.317
epoch: 10/100, loss: 1.314
epoch: 11/100, loss: 1.312
epoch: 12/100, loss: 1.31
epoch: 13/100, loss: 1.307
epoch: 14/100, loss: 1.305
epoch: 15/100, loss: 1.303
epoch: 16/100, loss: 1.301
epoch: 17/100, loss: 1.298
epoch: 18/100, loss: 1.296
epoch: 19/100, loss: 1.294
epoch: 20/100, loss: 1.292
epoch: 21/100, loss: 1.289
epoch: 22/100, loss: 1.287
epoch: 23/100, loss: 1.285
epoch: 24/100, loss: 1.282
epoch: 25/100, loss: 1.28
epoch: 26/100, loss: 1.278
epoch: 27/100, loss: 1.275
epoch: 28/100, loss: 1.273
epoch: 29/100, loss: 1.27
epoch: 30/100, loss: 1.268
epoch: 31/100, loss: 1.265
epoch: 32/100, loss: 1.263
epoch: 33/100, loss: 1.26
epoch: 34/100, loss: 1.257
epoch: 35/100, loss: 1.255
epoch: 36/100, loss: 1.252
epoch: 37/100, loss: 1.249
epoch: 38/100,

In [150]:
model(X_test)

tensor([[2.1447],
        [1.8328],
        [1.9868],
        ...,
        [2.4610],
        [2.0374],
        [2.0670]], grad_fn=<AddmmBackward0>)

In [110]:
ps = list(model.parameters())
w1 = ps[0]
b1 = ps[1]
w2 = ps[2]
p2 = ps[3]
w3 = ps[4]
b3 = ps[5]

In [112]:
w1.shape

torch.Size([50, 8])

In [114]:
X_train.shape

torch.Size([16512, 8])

In [132]:
X_train.shape

torch.Size([16512, 8])

In [130]:
w1.shape

torch.Size([50, 8])

In [135]:
(X_train @ w1.T + b1).shape

torch.Size([16512, 50])

In [128]:
b1

Parameter containing:
tensor([ 0.2933, -0.2330, -0.2869,  0.2687, -0.1687,  0.0230,  0.3522, -0.1411,
         0.3293,  0.2809,  0.2731,  0.1049, -0.1594,  0.2226, -0.2422, -0.2060,
        -0.1704,  0.1529,  0.0487,  0.2249,  0.2324,  0.0206,  0.2061, -0.2555,
        -0.3379, -0.2880,  0.1951,  0.3251, -0.0978,  0.1956, -0.2526, -0.0067,
        -0.0021, -0.1024, -0.1714,  0.1659, -0.0308, -0.0701,  0.2456, -0.2685,
         0.2309,  0.3140, -0.2172, -0.3350,  0.0492, -0.2689,  0.1478, -0.2820,
        -0.2759,  0.0956], requires_grad=True)